# TX resonant keying: overdrive vs keying speed -> rail / inductor / SMPS

The PRODUCT-generation drive (battery, low voltage, high-Q resonance), the one
the acoustic-frontend-architecture note explicitly defers out of the dev board.
Dev board brute-forces a +60 V HV rail through an STHV748 so the pins see the full
swing. Here we instead buy the transducer voltage from resonant Q magnification and
a low rail, then ask the question that decides the whole topology:

  the SAME loaded Q sets the voltage step-up, the bandwidth, AND the ring time,
  so how much rail headroom (overdrive) do we spend to key faster than Q allows,
  and what rail / inductor / SMPS does that pin down for a LOW-bitrate OOK link?

Model (single source of truth for transducer constants is rx_frontend / the
architecture note):
  - 1-bit (square) full-bridge drive at f0 = 90 kHz into a series-resonant tank.
  - Tank = matching inductor L in series with the transducer static capacitance
    C0 = 5 nF; loss/radiation lumped into one series R. Loaded Q = Z0 / R with
    Z0 = sqrt(L/C0). This is the project's working model (architecture note,
    "resonant voltage step-up"): one Q governs step-up, bandwidth, and ring.

Caveat carried throughout: the transducer is really Butterworth-Van Dyke (motional
Lm-Cm-Rm in parallel with C0), so "L resonates C0" and "the mechanical resonance
rings at Q_mech ~ 10" are two coupled resonances, not one. The single-Q series-RLC
here is the standard tractable reduction; it is honest for sizing and for the
ring-time / step-up TRADE, but treat absolute step-up as good to a factor, not a
decimal. The water-loaded Q is itself only ~2-10 (architecture note), and that
spread, not the third digit of any formula, is what dominates.

In [1]:
import numpy as np
import matplotlib.pyplot as plt

# --- transducer + drive constants (from rx_frontend / architecture note) ---
F0 = 90e3                      # Hz, resonance
W0 = 2 * np.pi * F0            # rad/s
C0 = 5.0e-9                    # F, static capacitance (Cs), +/-20%
RR = 150.0                     # ohm, in-air loss resistance (Rr, upper bound)
V_TARGET_RMS = 30.0            # Vrms across PZT, cavitation target (20 Vrms operating)

# loaded-Q operating points to carry through (water-loaded is 2-10, architecture note)
Q_GRID = (2.0, 5.0, 10.0)

# square-wave fundamental: full bridge swings +/-Vrail, fundamental peak = (4/pi) Vrail
K_FULLBRIDGE = 4 / np.pi


def L_match(C=C0, f=F0):
    """Series inductor that resonates C0 at f0."""
    return 1.0 / ((2 * np.pi * f) ** 2 * C)


L = L_match()
Z0 = np.sqrt(L / C0)            # tank characteristic impedance
print(f"L_match  = {L*1e6:6.1f} uH   (resonates {C0*1e9:.0f} nF at {F0/1e3:.0f} kHz)")
print(f"Z0       = {Z0:6.1f} ohm  = Xc0 = Xl0 at f0")
print(f"R_series for Q: " + "  ".join(f"Q{q:.0f}->{Z0/q:5.1f}ohm" for q in Q_GRID))
print(f"  (in-air Rr<{RR:.0f} ohm => bare-element loaded Q ~ {Z0/RR:.1f}; "
      f"matching/coupling is what lifts it)")

L_match  =  625.4 uH   (resonates 5 nF at 90 kHz)
Z0       =  353.7 ohm  = Xc0 = Xl0 at f0
R_series for Q: Q2->176.8ohm  Q5-> 70.7ohm  Q10-> 35.4ohm
  (in-air Rr<150 ohm => bare-element loaded Q ~ 2.4; matching/coupling is what lifts it)


## 1. Steady state: what rail reaches the target, and at what current

Series resonance magnifies the drive fundamental by Q at the capacitor (the
transducer terminal): V_pzt_pk = Q * V_drive_pk. The loop current is set by R, and
is the SAME current the bridge FETs carry, so a low rail does not mean low current,
the tank trades the voltage problem for a current problem (architecture note).

  V_pzt_pk    = Q * (4/pi) * Vrail
  Vrail(req)  = V_pzt_pk / (Q * 4/pi)            [for the target terminal voltage]
  I_tank_pk   = V_pzt_pk / Z0 = w0 * C0 * V_pzt_pk   [through L, C0, R and the FETs]
  P_real      = V_pzt_rms^2 / (Q^2 R) ... delivered into R (radiation + loss)

In [2]:
v_pzt_pk = V_TARGET_RMS * np.sqrt(2)
print(f"target {V_TARGET_RMS:.0f} Vrms = {v_pzt_pk:.1f} Vpk across the transducer\n")
print(f"{'Q':>4} {'Vrail req':>10} {'1S/2S?':>8} {'I_tank pk':>10} {'P_into_R':>9}")
for q in Q_GRID:
    vrail = v_pzt_pk / (q * K_FULLBRIDGE)
    i_pk = v_pzt_pk / Z0                      # independent of Q (set by terminal V)
    r_series = Z0 / q
    p_real = (V_TARGET_RMS ** 2) / (q ** 2 * r_series)  # = Vrms^2 * R / (Q R)^2... see note
    # delivered real power: I_rms^2 * R, I_rms = (V_pzt_pk/Z0)/sqrt2
    i_rms = i_pk / np.sqrt(2)
    p_real = i_rms ** 2 * r_series
    src = "1S" if vrail <= 3.7 else ("2S" if vrail <= 7.4 else "boost")
    print(f"{q:>4.0f} {vrail:>9.2f}V {src:>8} {i_pk*1e3:>8.0f}mA {p_real:>8.2f}W")
print("\n=> at Q=10 a bare 1S Li-ion (3.7 V) full bridge already reaches 30 Vrms;")
print("   the tank current ~120 mA is set by C0's reactance, not the rail. Low V, low I.")

target 30 Vrms = 42.4 Vpk across the transducer

   Q  Vrail req   1S/2S?  I_tank pk  P_into_R
   2     16.66V    boost      120mA     1.27W
   5      6.66V       2S      120mA     0.51W
  10      3.33V       1S      120mA     0.25W

=> at Q=10 a bare 1S Li-ion (3.7 V) full bridge already reaches 30 Vrms;
   the tank current ~120 mA is set by C0's reactance, not the rail. Low V, low I.


## 2. The Q tension: step-up and radiated power pull opposite ways

The loaded Q is not free to choose for step-up alone. R is dominated by RADIATION
(we WANT to radiate), so high Q = low R = little radiated power, and low Q = strong
radiation but little voltage gain. For a fixed terminal voltage the delivered real
power scales as 1/Q (above), so chasing step-up directly throttles the acoustic
output. This is why "free" voltage gain is modest and why the water-loaded Q lands
at 2-10, not 100.

In [3]:
q = np.linspace(1.5, 15, 200)
vrail_req = v_pzt_pk / (q * K_FULLBRIDGE)
p_real = (v_pzt_pk / Z0 / np.sqrt(2)) ** 2 * (Z0 / q)
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(q, vrail_req)
ax[0].axhline(3.7, ls="--", c="k", alpha=.5); ax[0].axhline(7.4, ls=":", c="k", alpha=.5)
ax[0].set(xlabel="loaded Q", ylabel="Vrail for 30 Vrms (full bridge)",
          title="step-up: higher Q -> lower rail")
ax[0].annotate("1S 3.7V", (12, 3.9)); ax[0].annotate("2S 7.4V", (12, 7.6))
ax[1].plot(q, p_real)
ax[1].set(xlabel="loaded Q", ylabel="real power into R at 30 Vrms (W)",
          title="...but radiated power falls as 1/Q")
for a in ax:
    a.grid(True, alpha=.3)

## 3. Ring dynamics: the cost of Q is keying speed

A resonator driven on resonance has an amplitude envelope with time constant
  tau = 2Q/w0 = Q/(pi f0)        [amplitude; energy decays in half that]
In cycles, tau * f0 = Q/pi. Passive ON needs ~5 tau to ring up to 99%, and OFF needs
~5 tau to decay; both edges cost the same. The -3 dB bandwidth f0/Q is the same fact
in the frequency domain.

In [4]:
print(f"{'Q':>4} {'tau':>9} {'cyc/tau':>8} {'5tau up':>9} {'BW=f0/Q':>9}")
for qq in Q_GRID:
    tau = qq / (np.pi * F0)
    print(f"{qq:>4.0f} {tau*1e6:>7.1f}us {tau*F0:>8.2f} {5*tau*1e6:>7.0f}us {F0/qq/1e3:>7.1f}kHz")
print("\n=> passive OOK symbol ~ 5tau(up) + dwell + 5tau(down). At Q=10 that's ~350 us")
print("   of edges alone -> a few kbaud ceiling before any shaping.")

   Q       tau  cyc/tau   5tau up   BW=f0/Q
   2     7.1us     0.64      35us    45.0kHz
   5    17.7us     1.59      88us    18.0kHz
  10    35.4us     3.18     177us     9.0kHz

=> passive OOK symbol ~ 5tau(up) + dwell + 5tau(down). At Q=10 that's ~350 us
   of edges alone -> a few kbaud ceiling before any shaping.


## 4. Overdrive ring-up and active-brake ring-down (beating Q)

The "5 tau" is only the PASSIVE envelope. With control authority we beat it both ways:

- **Overdrive ring-up.** Drive toward a VIRTUAL steady state m x higher than the hold
  target (m = rail headroom = Vrail_avail / Vrail_hold). Envelope rises as
  A(t) = m A_t (1 - e^{-t/tau}); it reaches the target A_t when
      t_up = tau * ln( m / (m-1) ).
  m=1 -> infinite (asymptotic); m=2 -> 0.69 tau; m=3 -> 0.41 tau; m=5 -> 0.22 tau.
  This is what the rail headroom BUYS. Needs voltage margin to inject the extra energy.

- **Active-brake ring-down.** Drive ANTI-PHASE to pull energy back out instead of
  waiting for losses. From A_t toward virtual -m A_t,
      t_down = tau * ln( (1+m) / m ) = tau * ln(1 + 1/m).
  m=1 -> 0.69 tau; m=2 -> 0.41 tau. Note m=1 already gives 0.69 tau vs ~4.6 tau passive
  decay-to-1%: braking is a ~7x win that needs NO extra rail headroom (same rail,
  opposite phase). Ring-down is the worse offender for OOK inter-symbol smear, so this
  is the cheap, high-value half.

In [5]:
def t_up(m, tau):    # cycles-equiv if tau in cycles
    return tau * np.log(m / (m - 1)) if m > 1 else np.inf


def t_down(m, tau):
    return tau * np.log(1 + 1 / m)


print("ring-up / ring-down in units of tau vs rail headroom m:")
print(f"{'m':>4} {'t_up/tau':>9} {'t_down/tau':>11}   ('passive' up~5.0 down~4.6)")
for m in (1, 1.5, 2, 3, 5, 10):
    tu = t_up(m, 1.0)
    tu_s = " passive" if not np.isfinite(tu) else f"{tu:.2f}"
    print(f"{m:>4.1f} {tu_s:>9} {t_down(m, 1.0):>11.2f}")

# envelope plot at Q=10
tau = 10 / (np.pi * F0)
t = np.linspace(0, 6 * tau, 800)
plt.figure(figsize=(7, 4))
for m, c_ in [(1, "C0"), (2, "C1"), (3, "C2")]:
    up = np.minimum(m * (1 - np.exp(-t / tau)), 1.0)   # clamp at hold target
    plt.plot(t * 1e6, up, c_, label=f"ring-up m={m}")
# braking ring-down from steady (m=2), starting at t=0 for overlay
dn = np.clip((1 + 2) * np.exp(-t / tau) - 2, 0, 1)
plt.plot(t * 1e6, dn, "C3--", label="brake-down m=2")
pas = np.exp(-t / tau)
plt.plot(t * 1e6, pas, "k:", alpha=.5, label="passive decay")
plt.axhline(1, ls="--", c="gray", alpha=.4)
plt.xlabel("time (us)"); plt.ylabel("envelope / target")
plt.title(f"Q=10, tau={tau*1e6:.0f} us: overdrive ring-up, active brake-down")
plt.legend(); plt.grid(True, alpha=.3)

ring-up / ring-down in units of tau vs rail headroom m:
   m  t_up/tau  t_down/tau   ('passive' up~5.0 down~4.6)
 1.0   passive        0.69
 1.5      1.10        0.51
 2.0      0.69        0.41
 3.0      0.41        0.29
 5.0      0.22        0.18
10.0      0.11        0.10


## 5. Achievable OOK symbol rate vs Q and headroom

Symbol = t_up + hold + t_down. Take a minimal hold of ~1 tau (enough cycles for the
receiver matched filter to integrate a clean tone) and combine overdrive ring-up with
active-brake ring-down. Convert to a baud ceiling. "Relatively low bitrate" for this
link means <~2 kbaud (the band is only ~9 kHz wide at Q=10 anyway), so the point is
to show how much margin we have, not to chase the ceiling.

In [6]:
HOLD_TAU = 1.0
print(f"{'Q':>4} " + " ".join(f"m={m:g}".rjust(9) for m in (1, 2, 3)) + "   (baud ceiling)")
for qq in Q_GRID:
    tau = qq / (np.pi * F0)
    row = []
    for m in (1, 2, 3):
        tu = 5 * tau if m == 1 else t_up(m, tau)      # m=1 passive ~5tau
        td = t_down(m, tau)
        sym = tu + HOLD_TAU * tau + td
        row.append(f"{1/sym:8.0f}")
    print(f"{qq:>4.0f} " + " ".join(r.rjust(9) for r in row))
print("\n=> even Q=10 / m=1 (no overdrive, no boost) clears ~2 kbaud; m=2-3 lifts it")
print("   ~5x. Low-bitrate target is met with comfort to spare at every Q.")

   Q       m=1       m=2       m=3   (baud ceiling)
   2     21122     67364     83496
   5      8449     26946     33399
  10      4224     13473     16699

=> even Q=10 / m=1 (no overdrive, no boost) clears ~2 kbaud; m=2-3 lifts it
   ~5x. Low-bitrate target is met with comfort to spare at every Q.


## 6. Practical pin-downs the analysis forces

**Tank mistune (the real practical risk).** C0 is +/-20%; resonance scales 1/sqrt(LC)
so that is +/-10% in f0. At Q=10 the band is only ~10% wide (9 kHz), so a worst-case
part sits at the band EDGE and loses most of the step-up. Either trim/bin L per unit,
or deliberately run a LOWER Q (5 -> 18 kHz band) to swallow the tolerance. This pushes
the recommendation toward Q~5, which also restores radiated power (section 2).

In [7]:
for tol in (0.20, 0.10):
    df = 0.5 * tol                      # f0 fractional shift ~ half the C tolerance
    for qq in (10.0, 5.0):
        bw = (1 / qq)                   # fractional -3 dB bandwidth
        print(f"C0 +/-{tol*100:.0f}% -> f0 +/-{df*100:.0f}%   vs Q={qq:.0f} band +/-{bw/2*100:.0f}% "
              f"-> {'EDGE/outside' if df > bw/2 else 'inside'} the passband")

C0 +/-20% -> f0 +/-10%   vs Q=10 band +/-5% -> EDGE/outside the passband
C0 +/-20% -> f0 +/-10%   vs Q=5 band +/-10% -> inside the passband
C0 +/-10% -> f0 +/-5%   vs Q=10 band +/-5% -> inside the passband
C0 +/-10% -> f0 +/-5%   vs Q=5 band +/-10% -> inside the passband


**Inductor.** ~625 uH carrying ~0.12-0.35 A (steady to overdrive transient), and its
ESR adds directly to R_series so it caps Q: for Q=10 the whole loop R is ~35 ohm, so
want ESR << that (a few ohm), easy for a 625 uH part at 90 kHz. Saturation current
only needs to cover the overdrive peak (<~0.5 A), tiny.

**Bridge.** Full bridge for +/-Vrail (2x swing, the differential-load win, architecture
note "full vs half bridge"). FET ratings are trivial: <~15 V, <~0.5 A. Anti-phase
braking is just the bridge's other diagonal, so active brake-down costs zero extra
silicon, only firmware.

## 7. Recommendation (low-bitrate battery product)

**Operating point: loaded Q ~ 5, full-bridge 1-bit at 90 kHz, adjustable boost rail
~12-15 V from 1S/2S Li-ion.**

Reasoning:
- **Q~5, not 10.** 10 maximises step-up but puts the +/-20% C0 spread at the band edge
  and starves radiated power. Q~5 keeps an ~18 kHz band that swallows the tolerance,
  restores acoustic output, and still gives ~6x voltage gain. Bin/trim the inductor
  only if bench data says Q~5 is still too tight.
- **Rail ~12-15 V (boost), not bare 2S.** Q~5 needs ~6.7 V to HOLD 30 Vrms, so 2S would
  run at m=1 (slow keying, no amplitude control). A small adjustable boost to ~12-15 V
  gives m~2-3 of overdrive headroom AND makes the rail the amplitude/TX-power knob
  (range and battery-life management), which is independently worth having. Burst-mode
  average power is sub-watt; size for average + a local bulk cap for the reactive ping,
  exactly the dev-board reservoir pattern.
- **Keying: overdrive ring-up + active-brake ring-down, hold via the rail level.** The
  brake is free (firmware-only, ~7x ring-down win, kills OOK smear); the overdrive
  spends the headroom for a ~5x faster ON edge. Section 5 shows this clears the
  low-bitrate target (<~2 kbaud) by a wide margin at Q~5.
- **L ~ 625 uH, low ESR, socketed/binned** to land on the actual measured C0, the one
  component the tolerance analysis says must be trimmed.

Falls back cleanly: drop the boost (run 2S direct, m=1) and the link still closes at
low bitrate, the boost is the upgrade that buys keying speed + amplitude control, not
a requirement. This is the low-voltage mirror of the dev board's brute-force HV rail:
same 30 Vrms at the transducer, ~13 V and ~0.3 A at the bridge instead of 60 V.

In [8]:
print("RECOMMENDATION (low-bitrate battery product)")
print("-" * 60)
qd = 5.0
vrail_hold = (V_TARGET_RMS * np.sqrt(2)) / (qd * K_FULLBRIDGE)
print(f"  topology    : full-bridge 1-bit @ {F0/1e3:.0f} kHz into series-L tank")
print(f"  loaded Q    : ~{qd:.0f}  (band ~{F0/qd/1e3:.0f} kHz, swallows +/-20% C0)")
print(f"  L_match     : ~{L*1e6:.0f} uH (socket/bin to measured C0), ESR<<{Z0/qd:.0f} ohm")
print(f"  Vrail hold  : ~{vrail_hold:.1f} V for 30 Vrms; boost to ~12-15 V for m~2-3 headroom")
print(f"  bridge FETs : <~15 V, <~0.5 A  (vs dev board 60 V HV rail)")
print(f"  keying      : overdrive ring-up + anti-phase brake-down; rail = amplitude knob")
print(f"  bitrate     : <~2 kbaud target, met with >5x margin (section 5)")
print(f"  SMPS        : small adjustable boost, burst-mode avg <1 W + local bulk cap")

RECOMMENDATION (low-bitrate battery product)
------------------------------------------------------------
  topology    : full-bridge 1-bit @ 90 kHz into series-L tank
  loaded Q    : ~5  (band ~18 kHz, swallows +/-20% C0)
  L_match     : ~625 uH (socket/bin to measured C0), ESR<<71 ohm
  Vrail hold  : ~6.7 V for 30 Vrms; boost to ~12-15 V for m~2-3 headroom
  bridge FETs : <~15 V, <~0.5 A  (vs dev board 60 V HV rail)
  keying      : overdrive ring-up + anti-phase brake-down; rail = amplitude knob
  bitrate     : <~2 kbaud target, met with >5x margin (section 5)
  SMPS        : small adjustable boost, burst-mode avg <1 W + local bulk cap
